# 🔁 Notebook 5 — Repetition Penalty: The Anti-Echo Chamber

> **The vibe:** Imagine you're in a room where every time you say a word, that word gets slightly harder to say again. Say it 5 times and it becomes almost unsayable. That's repetition penalty.  
> **Key word:** DISCOURAGES — it never bans a token, just makes it harder to repeat  
> **Time:** ~10 minutes


In [2]:
# ── Setup ────────────────────────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','text.color':'#e6edf3','axes.labelcolor':'#8b949e',
    'xtick.color':'#8b949e','ytick.color':'#8b949e','grid.color':'#21262d',
    'axes.grid':True,'grid.linewidth':0.5,'font.family':'monospace'})

TOKENS = ['approve','reject','review','escalate','delay','audit','optimize','notify','assign','close']
LOGITS = np.array([2.2,1.8,1.4,0.9,0.2,0.1,-0.3,-0.6,-0.8,-1.0])

def softmax(x): e=np.exp(x-x.max()); return e/e.sum()

def apply_rep_penalty(logits, penalty, seen_tokens):
    logits = logits.copy()
    for tok in seen_tokens:
        if logits[tok] > 0: logits[tok] /= penalty
        else:               logits[tok] *= penalty
    return logits

# Simulate a greedy loop — what happens without penalty
print("🔁 Without penalty — what a loop looks like:")
print("   approve → approve → approve → approve → approve → (forever)")
print("   (Model gets stuck because 'approve' always has the highest logit!)")
print()
print("🛡️  With penalty=1.3 — what breaks the loop:")
loop_seq = []
logits_loop = LOGITS.copy()
seen = []
for step in range(8):
    penalized = apply_rep_penalty(logits_loop, 1.3, seen)
    probs = softmax(penalized)
    chosen = np.argmax(probs)  # greedy for demo
    loop_seq.append(TOKENS[chosen])
    seen.append(chosen)
    if len(seen) > 4: seen = seen[-4:]  # lookback=4
print("   " + " → ".join(loop_seq))


🔁 Without penalty — what a loop looks like:
   approve → approve → approve → approve → approve → (forever)
   (Model gets stuck because 'approve' always has the highest logit!)

🛡️  With penalty=1.3 — what breaks the loop:
   approve → reject → approve → review → reject → approve → reject → approve


## 🎬 Graph 1 — The Echo Chamber vs The Anti-Echo Chamber

Left side: NO penalty. Model keeps looping "approve".  
Right side: WITH penalty. Each time "approve" is used, it gets harder to use again.

Watch the green bar (approve) shrink with each repetition.


In [1]:
# ── Graph 1: The Echo Chamber Effect ────────────────────────────────
LOOKBACK = 4
PENALTY  = 1.5

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('🔁  Echo Chamber (top) vs Anti-Echo Chamber (bottom)',
             fontsize=14, fontweight='bold', color='#ffd93d', y=1.02)

# Simulate without and with penalty
for row, (use_penalty, row_title, row_col) in enumerate([
    (False, '❌ No Penalty (penalty=1.0) — Notice how approve dominates EVERY step', '#ff6b9d'),
    (True,  f'✅ With Penalty={PENALTY} — approve gets pushed down, others get chances!', '#a8e6cf'),
]):
    seen = []
    fig.text(0.01, 0.77 - row*0.47, row_title, color=row_col, fontsize=10, fontweight='bold')
    for step in range(5):
        ax = axes[row][step]
        logits_step = apply_rep_penalty(LOGITS, PENALTY if use_penalty else 1.0, seen)
        probs_step  = softmax(logits_step)
        cols = ['#ff6b9d' if i in seen else '#a8e6cf' if not use_penalty else '#74b9ff'
                for i in range(10)]
        bars = ax.bar(range(10), probs_step, color=cols, alpha=0.85, width=0.7)
        top  = np.argmax(probs_step)
        bars[top].set_edgecolor('white'); bars[top].set_linewidth(2)
        for s in seen: bars[s].set_color('#ff6b9d'); bars[s].set_alpha(0.4)
        ax.set_xticks(range(10)); ax.set_xticklabels(TOKENS, rotation=45, ha='right', fontsize=6)
        ax.set_ylim(0, 1.05)
        chosen_tok = TOKENS[top]
        ax.set_title(f'Step {step+1}
Chooses: "{chosen_tok}"',
                    color='white' if chosen_tok not in [TOKENS[s] for s in seen] else '#ff6b9d',
                    fontsize=9, fontweight='bold')
        seen.append(top)
        if len(seen) > LOOKBACK: seen = seen[-LOOKBACK:]

plt.tight_layout()
plt.savefig('graph1_rep_echo.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n💡 Red bars = tokens that have been used recently (penalized)")
print(f"💡 With penalty={PENALTY}, approve's probability drops each time it's used")


SyntaxError: unterminated f-string literal (detected at line 29) (3613936483.py, line 29)

## 🎬 Graph 2 — Penalty Strength: From Whisper to Shout

How much does each penalty value actually change probabilities?

- **1.0**: Complete silence (no effect)  
- **1.1**: A whisper (tiny nudge)  
- **1.5**: A firm push  
- **2.0**: A shout (hard to repeat)  
- **3.0**: Nearly impossible to repeat


In [ ]:
# ── Graph 2: Penalty Strength Sweep ─────────────────────────────────
# Simulate: 'approve' (logit 2.2) has already been generated once
SEEN_TOKENS = [0]  # approve is index 0
penalties = [1.0, 1.1, 1.3, 1.5, 2.0, 3.0]
pen_colors = ['#8b949e','#a8e6cf','#74b9ff','#ffd93d','#ff9f43','#ff6b9d']

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
fig.suptitle('📣  Penalty Strength: From Whisper to Shout (approve was already used once)',
             fontsize=12, fontweight='bold', color='#ffd93d', y=1.02)

for ax, pen, col in zip(axes, penalties, pen_colors):
    penalized_logits = apply_rep_penalty(LOGITS, pen, SEEN_TOKENS)
    probs = softmax(penalized_logits)
    bar_c = [col if i in SEEN_TOKENS else '#2d333b' for i in range(10)]
    bars  = ax.bar(range(10), probs, color=bar_c, alpha=0.9, width=0.7)
    top   = np.argmax(probs)
    if top not in SEEN_TOKENS:
        bars[top].set_edgecolor('white'); bars[top].set_linewidth(2)
    ent = -np.sum(probs * np.log(probs + 1e-12))
    ax.set_xticks(range(10)); ax.set_xticklabels(TOKENS, rotation=45, ha='right', fontsize=6)
    ax.set_ylim(0, max(softmax(LOGITS))*1.15)
    label = {1.0:'🤫 Silent',1.1:'🔈 Whisper',1.3:'🔉 Nudge',
             1.5:'🔊 Push',2.0:'📣 Shout',3.0:'🔇 Mute!'}[pen]
    ax.set_title(f'{label}
pen={pen}
approve={probs[0]:.1%}',
                color=col, fontsize=8, fontweight='bold')
    ax.set_ylabel('Prob' if ax==axes[0] else '')

plt.tight_layout()
plt.savefig('graph2_rep_strength.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\nPenalty effect on 'approve' probability:")
original_p = softmax(LOGITS)[0]
print(f"  Original (no penalty): {original_p:.1%}")
for pen in penalties:
    new_p = softmax(apply_rep_penalty(LOGITS, pen, [0]))[0]
    change = new_p - original_p
    bar = '▼' * int(abs(change)*30)
    print(f"  Penalty={pen}: {new_p:.1%}  {bar} ({change:+.1%})")


## 🎬 Graph 3 — Presence vs Frequency: Two Different Taxes

**Presence penalty:** Pay a flat tax once if you've appeared at all.  
**Frequency penalty:** Pay more each time you appear. The 5th appearance costs 5x more than the 1st.


In [ ]:
# ── Graph 3: Presence vs Frequency Penalty ───────────────────────────
appearances = {0: 4, 1: 2, 5: 1}  # token_idx: how many times appeared

def presence_penalty_fn(logits, penalty_val, seen):
    l = logits.copy()
    for t in seen: l[t] -= penalty_val
    return l

def frequency_penalty_fn(logits, penalty_val, counts):
    l = logits.copy()
    for t, cnt in counts.items(): l[t] -= penalty_val * cnt
    return l

STRENGTH = 0.5
tokens_to_show = list(appearances.keys())
appear_counts  = list(appearances.values())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('⚖️  Presence vs Frequency Penalty: Two Different Taxes',
             fontsize=14, fontweight='bold', color='#ffd93d')

# Original
orig_probs = softmax(LOGITS)
axes[0].bar(range(10), orig_probs, color='#8b949e', alpha=0.7, width=0.7)
for idx, cnt in appearances.items():
    axes[0].bar(idx, orig_probs[idx], color='#ffd93d', alpha=0.9, width=0.7)
    axes[0].text(idx, orig_probs[idx]+0.005, f'seen
{cnt}x', ha='center', va='bottom',
                color='#ffd93d', fontsize=8, fontweight='bold')
axes[0].set_xticks(range(10)); axes[0].set_xticklabels(TOKENS, rotation=45, ha='right', fontsize=7)
axes[0].set_title('Original Distribution
(highlighted = seen tokens)', color='#ffd93d', fontweight='bold')

pres_logits = presence_penalty_fn(LOGITS, STRENGTH, list(appearances.keys()))
pres_probs  = softmax(pres_logits)
freq_logits = frequency_penalty_fn(LOGITS, STRENGTH, appearances)
freq_probs  = softmax(freq_logits)

for ax, probs, title, col in [
    (axes[1], pres_probs, f'Presence Penalty (λ={STRENGTH})
Flat tax: all seen tokens = same penalty', '#ff9f43'),
    (axes[2], freq_probs, f'Frequency Penalty (λ={STRENGTH})
Escalating tax: more appearances = heavier penalty', '#ff6b9d'),
]:
    ax.bar(range(10), orig_probs, color='#2d333b', alpha=0.5, width=0.7, label='Original')
    ax.bar(range(10), probs, color=col, alpha=0.8, width=0.5, label='After penalty')
    for idx, cnt in appearances.items():
        change = probs[idx] - orig_probs[idx]
        ax.annotate(f'{change:+.1%}', (idx, max(probs[idx], orig_probs[idx])),
                   textcoords='offset points', xytext=(0, 5),
                   ha='center', color='white', fontsize=8, fontweight='bold')
    ax.set_xticks(range(10)); ax.set_xticklabels(TOKENS, rotation=45, ha='right', fontsize=7)
    ax.set_title(title, color=col, fontweight='bold', fontsize=9)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('graph3_rep_presence_freq.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n💡 For 'approve' (seen 4x, index 0):")
print(f"   Original:          {orig_probs[0]:.1%}")
print(f"   After presence:    {pres_probs[0]:.1%}  (flat deduction regardless of count)")
print(f"   After frequency:   {freq_probs[0]:.1%}  (4× heavier deduction because seen 4×)")


## 🎮 Graph 4 — Playground: Your Anti-Echo Settings


In [ ]:
# ── Graph 4: Playground ──────────────────────────────────────────────
PENALTY   = 1.3   # ← 🎮 CHANGE THIS! (1.0=off, 1.1=light, 1.5=heavy, 2.0=extreme)
N_SEEN    = 2     # ← How many times has the top token already appeared?
LOOKBACK  = 4     # ← How many recent tokens to penalize?

seen_tokens = [0] * min(N_SEEN, LOOKBACK)  # approve appeared N_SEEN times

penalized_l = apply_rep_penalty(LOGITS, PENALTY, seen_tokens)
probs_orig  = softmax(LOGITS)
probs_pen   = softmax(penalized_l)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'🎮 Playground: penalty={PENALTY}, seen {N_SEEN}× in last {LOOKBACK} tokens',
             fontsize=14, fontweight='bold', color='#ffd93d')

x = np.arange(10)
axes[0].bar(x - 0.2, probs_orig, 0.4, color='#8b949e', alpha=0.8, label='Original')
axes[0].bar(x + 0.2, probs_pen,  0.4, color='#a8e6cf', alpha=0.9, label='After penalty')
for i in range(N_SEEN):
    axes[0].bar(0 - 0.2, probs_orig[0], 0.4, color='#ff6b9d', alpha=0.9)
    axes[0].bar(0 + 0.2, probs_pen[0],  0.4, color='#ffd93d', alpha=0.9)
axes[0].set_xticks(x); axes[0].set_xticklabels(TOKENS, rotation=45, ha='right', fontsize=7)
axes[0].legend(fontsize=9); axes[0].set_title('Before vs After', color='#a8e6cf', fontweight='bold')

pen_range = np.linspace(1.0, 3.0, 200)
approve_probs_over_pen = [softmax(apply_rep_penalty(LOGITS, p, [0]*min(N_SEEN, LOOKBACK)))[0] for p in pen_range]
axes[1].plot(pen_range, approve_probs_over_pen, color='#ff6b9d', lw=2.5)
axes[1].scatter([PENALTY], [probs_pen[0]], s=200, color='#ffd93d', zorder=10, edgecolors='white', lw=2)
axes[1].axhline(probs_orig[0], color='#8b949e', lw=1.5, ls='--', label=f'Original: {probs_orig[0]:.1%}')
axes[1].axvline(PENALTY, color='#ffd93d', lw=1.5, ls='--')
axes[1].set_xlabel('Penalty'); axes[1].set_ylabel("P('approve')")
axes[1].set_title(f"Approve's fate across penalty values
(seen {N_SEEN} times)", color='#ff6b9d', fontweight='bold')
axes[1].legend(fontsize=9)

# Safety gauge
axes[2].axis('off')
ent = -np.sum(probs_pen * np.log(probs_pen + 1e-12))
level = 'OFF' if PENALTY == 1.0 else 'SAFE ✅' if PENALTY <= 1.3 else 'STRONG ⚠️' if PENALTY <= 1.7 else 'DANGER ❌'
color_level = {'OFF':'#8b949e','SAFE ✅':'#a8e6cf','STRONG ⚠️':'#ffd93d','DANGER ❌':'#ff6b9d'}[level]
axes[2].text(0.5, 0.7, f'Penalty Level:

{level}', ha='center', va='center',
            transform=axes[2].transAxes, fontsize=20, color=color_level, fontweight='bold',
            bbox=dict(boxstyle='round,pad=1', facecolor='#21262d', edgecolor=color_level, lw=3))
axes[2].text(0.5, 0.2, f"approve: {probs_orig[0]:.1%} → {probs_pen[0]:.1%}
Entropy: {ent:.2f} bits",
            ha='center', va='center', transform=axes[2].transAxes, fontsize=12, color='white')

plt.tight_layout()
plt.savefig('graph4_rep_playground.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## ✅ Notebook 5 Complete

| Concept | In One Line |
|---|---|
| Repetition penalty | Divides positive logits / multiplies negative logits of seen tokens |
| Discourages, not forbids | Even penalized tokens can still be sampled |
| Lookback window | Only tokens in the last N positions are penalized |
| Presence penalty | Flat tax — same deduction regardless of how many times seen |
| Frequency penalty | Escalating tax — seen 5× = 5× heavier penalty |
| Safe range | 1.1–1.3 for most tasks; >1.8 risks incoherence |

> **🔁 Echo chamber metaphor holds:** The more you repeat a word, the harder it becomes to say it again — but it's never truly silenced.

**Next → `06_usecases.ipynb`: The Decision Flowchart 🗺️**
